In [23]:
import os
import logging
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
 
logging.getLogger("tensorflow").setLevel(logging.ERROR)
 
from transformers import AutoTokenizer
from transformers import TFAutoModel
from arabert.preprocess import ArabertPreprocessor
 
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [25]:
from symbolicModel import (
    build_symbolic_matrix,
    learn_rule_confidence,
    RULE_CONFIDENCE,
    symbolic_detector,
    show_symbolic_result
)

In [26]:
ModelName  = "aubmindlab/bert-large-arabertv02"
DATA_DIR   = "/Users/roamsaleh/Downloads/similedata/"
 
MAX_LEN    = 120
BATCH_SIZE = 16
SEED       = 42
 
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [27]:
train_df = pd.read_csv(DATA_DIR + "cleaned_train.csv")
val_df   = pd.read_csv(DATA_DIR + "cleaned_validation.csv")
test_df  = pd.read_csv(DATA_DIR + "cleaned_test.csv")
 
X_train = train_df["text"].tolist()
y_train = train_df["label"].tolist()
 
X_val = val_df["text"].tolist()
y_val = val_df["label"].tolist()
 
X_test = test_df["text"].tolist()
y_test = test_df["label"].tolist()
 
print(len(X_train), len(X_val), len(X_test))
 

6000 1108 3000


In [7]:
X_train = train_df["text"].tolist()
y_train = train_df["label"].tolist()
 
X_val = val_df["text"].tolist()
y_val = val_df["label"].tolist()
 
X_test = test_df["text"].tolist()
y_test = test_df["label"].tolist()
 
print(len(X_train), len(X_val), len(X_test))

6000 1108 3000


In [28]:
learned_scores = learn_rule_confidence(X_train, y_train)
RULE_CONFIDENCE.update(learned_scores)
 
print("\nLearned Rule Confidence:")
print(RULE_CONFIDENCE)


Learned Rule Confidence:
{'explicit_particle': 0.9, 'weak_particle': 0.983, 'verb_simile': 0.927, 'nominal_simile': 0.985, 'prefix_simile': 0.676}


In [29]:
X_train_sym = build_symbolic_matrix(X_train)
X_val_sym   = build_symbolic_matrix(X_val)
X_test_sym  = build_symbolic_matrix(X_test)
 
print("\nSymbolic Feature Shapes:")
print(X_train_sym.shape, X_val_sym.shape, X_test_sym.shape)


Symbolic Feature Shapes:
(6000, 11) (1108, 11) (3000, 11)


In [30]:
arabert_prep = ArabertPreprocessor(model_name=ModelName)
tokenizer    = AutoTokenizer.from_pretrained(ModelName)
 
def PreprocessTexts(texts):
    texts     = [arabert_prep.preprocess(t) for t in texts]
    encodings = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return encodings['input_ids'], encodings['attention_mask']
 
X_train_ids, X_train_mask = PreprocessTexts(X_train)
X_val_ids,   X_val_mask   = PreprocessTexts(X_val)
X_test_ids,  X_test_mask  = PreprocessTexts(X_test)

In [31]:
TRAIN_ATTN_CACHE = DATA_DIR + "train_cls_attn.npy"
VAL_ATTN_CACHE   = DATA_DIR + "val_cls_attn.npy"
TEST_ATTN_CACHE  = DATA_DIR + "test_cls_attn.npy"
 
print("\nLoading AraBERT for attention extraction...")
 
arabert_attention = TFAutoModel.from_pretrained(
    ModelName,
    output_attentions=True,
    from_pt=True
)
 
def extract_attention_patterns(input_ids, attention_mask,
                                batch_size=64, cache_path=None):
    """
    Extract [CLS] attention patterns from AraBERT.
    Averaged across all 24 layers and 16 heads.
    Returns np.array (N, MAX_LEN).
    Saves to cache_path so re-runs load instantly.
    """
    if cache_path and os.path.exists(cache_path):
        print(f"  Loading from cache: {cache_path}")
        return np.load(cache_path)
 
    all_cls_attn = []
    total        = input_ids.shape[0]
 
    for start in range(0, total, batch_size):
        end    = min(start + batch_size, total)
        b_ids  = input_ids[start:end]
        b_mask = attention_mask[start:end]
 
        outputs = arabert_attention(
            input_ids=b_ids,
            attention_mask=b_mask,
            training=False,
            return_dict=True
        )
 
        # (24, batch, heads, seq, seq)
        attn_stack = tf.stack(outputs.attentions, axis=0)
 
        # [CLS] attending to all positions → (24, batch, heads, seq)
        cls_attn = attn_stack[:, :, :, 0, :]
 
        # Average layers + heads → (batch, seq)
        cls_attn_mean = tf.reduce_mean(cls_attn, axis=[0, 2])
        all_cls_attn.append(cls_attn_mean.numpy())
 
        if (start // batch_size) % 10 == 0:
            print(f"  {end}/{total}  ({int(end/total*100)}%)")
 
    result = np.vstack(all_cls_attn)
 
    if cache_path:
        np.save(cache_path, result)
        print(f"  Saved to cache: {cache_path}")
 
    return result
 
 
print("\nExtracting attention from train set (runs once, then cached)...")
train_cls_attn = extract_attention_patterns(
    X_train_ids, X_train_mask,
    batch_size=64, cache_path=TRAIN_ATTN_CACHE
)
 
print("Extracting attention from val set...")
val_cls_attn = extract_attention_patterns(
    X_val_ids, X_val_mask,
    batch_size=64, cache_path=VAL_ATTN_CACHE
)
 
print("Extracting attention from test set...")
test_cls_attn = extract_attention_patterns(
    X_test_ids, X_test_mask,
    batch_size=64, cache_path=TEST_ATTN_CACHE
)
 
# =====================================================
# STEP 2 — MINE NEURAL RULES FROM ATTENTION
# =====================================================
 
def mine_neural_rules(input_ids, cls_attn, labels,
                       top_k=5, ratio_threshold=2.0):
    """
    Discover which tokens AraBERT attends to MORE
    in Simile sentences vs Not Simile sentences.
    Returns a dict of {token: likelihood_ratio}.
    """
    simile_counts     = defaultdict(float)
    not_simile_counts = defaultdict(float)
 
    for idx, (attn_row, label) in enumerate(zip(cls_attn, labels)):
        top_positions = np.argsort(attn_row)[-top_k:]
        for pos in top_positions:
            token_id  = int(input_ids[idx, pos].numpy())
            token_str = tokenizer.decode([token_id]).strip()
            if token_str in ['[CLS]', '[SEP]', '[PAD]', '']:
                continue
            if label == 1:
                simile_counts[token_str]     += attn_row[pos]
            else:
                not_simile_counts[token_str] += attn_row[pos]
 
    simile_total     = sum(simile_counts.values())     + 1e-8
    not_simile_total = sum(not_simile_counts.values()) + 1e-8
 
    likelihood_ratios = {}
    for token in set(simile_counts) | set(not_simile_counts):
        p_s  = simile_counts.get(token, 1e-8)     / simile_total
        p_ns = not_simile_counts.get(token, 1e-8) / not_simile_total
        likelihood_ratios[token] = p_s / p_ns
 
    discriminative = {
        t: r for t, r in likelihood_ratios.items()
        if r > ratio_threshold
    }
    discriminative = dict(
        sorted(discriminative.items(), key=lambda x: x[1], reverse=True)
    )
 
    print("\nNeural Rules Discovered from Attention:")
    for token, ratio in list(discriminative.items())[:15]:
        print(f"  → '{token}'  (likelihood ratio: {ratio:.2f}x)")
 
    return discriminative
 
 
print("\nMining neural rules from training attention...")
discriminative_tokens = mine_neural_rules(
    X_train_ids, train_cls_attn, y_train
)


Loading AraBERT for attention extraction...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the 


Extracting attention from train set (runs once, then cached)...
  Loading from cache: /Users/roamsaleh/Downloads/similedata/train_cls_attn.npy
Extracting attention from val set...
  Loading from cache: /Users/roamsaleh/Downloads/similedata/val_cls_attn.npy
Extracting attention from test set...
  Loading from cache: /Users/roamsaleh/Downloads/similedata/test_cls_attn.npy

Mining neural rules from training attention...

Neural Rules Discovered from Attention:
  → 'اشبه'  (likelihood ratio: 45571844.00x)
  → 'حقيقي'  (likelihood ratio: 44189176.00x)
  → 'كالم'  (likelihood ratio: 41965384.00x)
  → '##عي'  (likelihood ratio: 38248584.00x)
  → 'يشبه'  (likelihood ratio: 38158516.00x)
  → 'كال'  (likelihood ratio: 37420004.00x)
  → 'الطوب'  (likelihood ratio: 33651944.00x)
  → '##a'  (likelihood ratio: 31024418.00x)
  → 'كرجل'  (likelihood ratio: 30377986.00x)
  → 'كش'  (likelihood ratio: 28830286.00x)
  → 'كالا'  (likelihood ratio: 26200280.00x)
  → '##ما'  (likelihood ratio: 24040456.00x)

In [32]:
def build_neural_attention_features(input_ids, cls_attn,
                                     discriminative_tokens):
    disc_ids = set()
    for token in discriminative_tokens:
        for tid in tokenizer.encode(token, add_special_tokens=False):
            disc_ids.add(tid)
 
    features = []
    for idx, attn_row in enumerate(cls_attn):
 
        # Feature 0: max attention to discriminative tokens
        attn_to_key = 0.0
        for pos in range(attn_row.shape[0]):
            if int(input_ids[idx, pos].numpy()) in disc_ids:
                attn_to_key = max(attn_to_key, float(attn_row[pos]))
 
        # Feature 1: attention entropy (low = focused / confident)
        attn_norm = attn_row / (attn_row.sum() + 1e-10)
        entropy   = float(-np.sum(attn_norm * np.log(attn_norm + 1e-10)))
 
        # Feature 2: early vs late position bias
        mid      = attn_row.shape[0] // 2
        pos_bias = float(attn_row[:mid].mean() - attn_row[mid:].mean())
 
        features.append([attn_to_key, entropy, pos_bias])
 
    return np.array(features, dtype=np.float32)
 
 
print("\nBuilding neural attention features...")
train_neural_feats = build_neural_attention_features(
    X_train_ids, train_cls_attn, discriminative_tokens
)
val_neural_feats = build_neural_attention_features(
    X_val_ids, val_cls_attn, discriminative_tokens
)
test_neural_feats = build_neural_attention_features(
    X_test_ids, test_cls_attn, discriminative_tokens
)
 


Building neural attention features...


In [33]:
X_train_combined = np.hstack([X_train_sym, train_neural_feats])
X_val_combined   = np.hstack([X_val_sym,   val_neural_feats])
X_test_combined  = np.hstack([X_test_sym,  test_neural_feats])
 
print("\nCombined Feature Shapes:")
print(X_train_combined.shape, X_val_combined.shape, X_test_combined.shape)


Combined Feature Shapes:
(6000, 14) (1108, 14) (3000, 14)


In [35]:
class HybridAraBERT(tf.keras.Model):
 
    def __init__(self, model_name, alpha=0.7):
        super().__init__()
 
        self.arabert = TFAutoModel.from_pretrained(
            model_name, from_pt=True
        )
        self.alpha = alpha
 
        # Neural classifier head
        self.dense1   = tf.keras.layers.Dense(
            256, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(0.01)
        )
        self.dropout  = tf.keras.layers.Dropout(0.5)
        self.dense2   = tf.keras.layers.Dense(64, activation='relu')
        self.bert_out = tf.keras.layers.Dense(
            1, activation='sigmoid', name="bert_probability"
        )
 
        # Symbolic sub-network — all 14 features
        self.sym_dense = tf.keras.layers.Dense(
            16, activation='relu', name="sym_hidden"
        )
        self.sym_out = tf.keras.layers.Dense(
            1, activation='sigmoid', name="sym_score"
        )
 
    def _forward(self, inputs, training=False):
        input_ids, attention_mask, combined_input = inputs
 
        outputs = self.arabert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            training=training,
            return_dict=False
        )[0]
 
        cls       = outputs[:, 0, :]
        x         = self.dense1(cls)
        x         = self.dropout(x, training=training)
        x         = self.dense2(x)
        bert_prob = self.bert_out(x)
 
        sym_hidden     = self.sym_dense(combined_input)
        symbolic_score = self.sym_out(sym_hidden)
 
        final_output = (
            self.alpha * bert_prob +
            (1 - self.alpha) * symbolic_score
        )
 
        return final_output, bert_prob
 
    def call(self, inputs, training=False):
        """Single output — required for Keras 3 training."""
        final_output, _ = self._forward(inputs, training=training)
        return final_output
 
    def predict_both(self, inputs):
        """Returns (final_output, bert_prob) for evaluation."""
        return self._forward(inputs, training=False)
 
 

In [36]:
model = HybridAraBERT(ModelName, alpha=0.7)
 
# Warm-up — builds all weights
_ = model(
    [X_train_ids[:1], X_train_mask[:1], X_train_combined[:1]],
    training=False
)
 
model.summary()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the 

Model: "hybrid_ara_bert_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 tf_bert_model_4 (TFBertMod  multiple                  369423360 
 el)                                                             
                                                                 
 dense_2 (Dense)             multiple                  262400    
                                                                 
 dropout_366 (Dropout)       multiple                  0         
                                                                 
 dense_3 (Dense)             multiple                  16448     
                                                                 
 bert_probability (Dense)    multiple                  65        
                                                                 
 sym_hidden (Dense)          multiple                  240       
                                                 

In [37]:
print("\n── Phase 1: Training classifier head (AraBERT frozen) ──")
 
model.arabert.trainable = False
 
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
 
early_stop_p1 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=2,
    restore_best_weights=True
)
 
history_p1 = model.fit(
    [X_train_ids, X_train_mask, X_train_combined],
    np.array(y_train),
    validation_data=(
        [X_val_ids, X_val_mask, X_val_combined],
        np.array(y_val)
    ),
    epochs=3,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop_p1]
)


── Phase 1: Training classifier head (AraBERT frozen) ──
Epoch 1/3
375/375 [==============================] - 1093s 3s/step - loss: 4.1001 - accuracy: 0.5537 - val_loss: 3.3747 - val_accuracy: 0.6679
Epoch 2/3
375/375 [==============================] - 1046s 3s/step - loss: 2.9269 - accuracy: 0.6470 - val_loss: 2.4256 - val_accuracy: 0.7608
Epoch 3/3
375/375 [==============================] - 1063s 3s/step - loss: 2.1523 - accuracy: 0.7132 - val_loss: 1.7927 - val_accuracy: 0.8132


In [38]:
print("\n── Phase 2: Fine-tuning full model (AraBERT unfrozen) ──")
 
model.arabert.trainable = True
 
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-6),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
 
early_stop_p2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=2,
    restore_best_weights=True
)
 
history_p2 = model.fit(
    [X_train_ids, X_train_mask, X_train_combined],
    np.array(y_train),
    validation_data=(
        [X_val_ids, X_val_mask, X_val_combined],
        np.array(y_val)
    ),
    epochs=7,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop_p2]
)
 
# Merge histories for plotting
history_merged = {}
for key in history_p1.history:
    history_merged[key] = (
        history_p1.history[key] + history_p2.history[key]
    )


── Phase 2: Fine-tuning full model (AraBERT unfrozen) ──
Epoch 1/7
375/375 [==============================] - 3185s 8s/step - loss: 1.7254 - accuracy: 0.8428 - val_loss: 1.5955 - val_accuracy: 0.9170
Epoch 2/7
375/375 [==============================] - 3146s 8s/step - loss: 1.6338 - accuracy: 0.8920 - val_loss: 1.5570 - val_accuracy: 0.9260
Epoch 3/7
375/375 [==============================] - 3178s 8s/step - loss: 1.5907 - accuracy: 0.9017 - val_loss: 1.5229 - val_accuracy: 0.9332
Epoch 4/7
375/375 [==============================] - 3163s 8s/step - loss: 1.5644 - accuracy: 0.9085 - val_loss: 1.4951 - val_accuracy: 0.9504
Epoch 5/7
375/375 [==============================] - 3096s 8s/step - loss: 1.5440 - accuracy: 0.9075 - val_loss: 1.4699 - val_accuracy: 0.9549
Epoch 6/7
375/375 [==============================] - 3119s 8s/step - loss: 1.5157 - accuracy: 0.9158 - val_loss: 1.4470 - val_accuracy: 0.9567
Epoch 7/7
375/375 [==============================] - 3088s 8s/step - loss: 1.4714 - 

In [1]:
y_val_pred_prob, bert_val_prob = model.predict_both(
    [X_val_ids, X_val_mask, X_val_combined]
)
 
y_val_pred = (y_val_pred_prob.numpy() > 0.5).astype(int)

NameError: name 'model' is not defined

In [1]:
print("\nValidation Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))
 
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred))


Validation Confusion Matrix:


NameError: name 'confusion_matrix' is not defined

In [ ]:
y_pred_prob, bert_test_prob = model.predict_both(
    [X_test_ids, X_test_mask, X_test_combined]
)
 
y_pred = (y_pred_prob.numpy() > 0.5).astype(int)

In [ ]:
print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
 
print("\nTest Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure()
 
plt.plot(history_merged['accuracy'],     label='Train Accuracy')
plt.plot(history_merged['val_accuracy'], label='Validation Accuracy')
plt.plot(history_merged['loss'],     linestyle='--', label='Train Loss')
plt.plot(history_merged['val_loss'], linestyle='--', label='Validation Loss')
 
# Mark the boundary between phase 1 and phase 2
phase1_epochs = len(history_p1.history['loss'])
plt.axvline(x=phase1_epochs - 1, color='gray',
            linestyle=':', label='Unfreeze AraBERT')
 
plt.xlabel("Epochs")
plt.ylabel("Value")
plt.title("Training vs Validation Performance")
plt.legend()
plt.show()

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
 
plt.figure()
plt.imshow(cm)
plt.title("Validation Confusion Matrix")
plt.colorbar()
 
classes    = ["Not Simile", "Simile"]
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
 
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
 
plt.show()
 

In [ ]:
def explain_prediction(sentence):
 
    print("=" * 65)
    print("SENTENCE:", sentence)
 
    symbolic_result = symbolic_detector(sentence)
 
    enc = tokenizer(
        [arabert_prep.preprocess(sentence)],
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    ids  = enc['input_ids']
    mask = enc['attention_mask']
 
    single_cls_attn = extract_attention_patterns(
        ids, mask, batch_size=1, cache_path=None
    )
 
    sym_feat    = build_symbolic_matrix([sentence])
    neural_feat = build_neural_attention_features(
        ids, single_cls_attn, discriminative_tokens
    )
    combined = np.hstack([sym_feat, neural_feat])
 
    final_prob, bert_prob = model.predict_both([ids, mask, combined])
 
    final_prob = float(final_prob[0][0])
    bert_prob  = float(bert_prob[0][0])
    prediction = "Simile ✓" if final_prob > 0.5 else "Not Simile ✗"
 
    # [1] Grammar layer
    print("\n[1] SYMBOLIC GRAMMAR LAYER")
    print(f"  has_structure       : {int(sym_feat[0][0])}")
    print(f"  num_structures      : {int(sym_feat[0][1])}")
    print(f"  max_conf            : {round(float(sym_feat[0][2]), 3)}")
    print(f"  avg_conf            : {round(float(sym_feat[0][3]), 3)}")
    print(f"  has_strong_particle : {int(sym_feat[0][4])}  (كأن / كأنّ)")
    print(f"  has_weak_particle   : {int(sym_feat[0][5])}  (كـ / مثل)")
    print(f"  has_verb            : {int(sym_feat[0][6])}  (يشبه / يماثل)")
    print(f"  has_noun            : {int(sym_feat[0][7])}  (شبيه / مثيل)")
    print(f"  has_prefix          : {int(sym_feat[0][8])}  (كـ prefix)")
    print(f"  avg_distance        : {round(float(sym_feat[0][9]), 3)}")
    print(f"  multi_simile_flag   : {int(sym_feat[0][10])}")
 
    if symbolic_result.structures:
        for s in symbolic_result.structures:
            print(f"\n  Rule: {s.rule}")
            print(f"  المشبه: {s.subject}  |  الأداة: {s.particle}  |  المشبه به: {s.object}")
            print(f"  Confidence: {round(s.confidence, 3)}")
    else:
        print("\n  No grammar structures detected")
 
    # [2] Neural attention layer
    print("\n[2] NEURAL ATTENTION LAYER")
    attn_to_key = float(neural_feat[0][0])
    entropy     = float(neural_feat[0][1])
    pos_bias    = float(neural_feat[0][2])
 
    print(f"  attn_to_key_tokens : {round(attn_to_key, 4)}")
    print(f"  attention_entropy  : {round(entropy, 4)}")
    print(f"  attention_pos_bias : {round(pos_bias, 4)}")
 
    attn_row  = single_cls_attn[0]
    top_5_pos = np.argsort(attn_row)[-5:][::-1]
 
    print("\n  Top tokens AraBERT attended to:")
    for pos in top_5_pos:
        token_str = tokenizer.decode(
            [int(ids[0, pos].numpy())]
        ).strip()
        tag = "← neural rule" if token_str in discriminative_tokens else ""
        print(f"    '{token_str}'  attn={attn_row[pos]:.4f}  {tag}")
 
    print("\n  Symbolic interpretation of AraBERT:")
    print("  →", "Focused on simile indicators (reliable)"
          if attn_to_key > 0.1
          else "Did not focus on known simile indicators")
    print("  →", "Attention focused — confident"
          if entropy < 4.0
          else ("Attention moderate — uncertain"
                if entropy < 4.5
                else "Attention scattered — very uncertain"))
    print("  →", "Early-token focus (normal for Arabic similes)"
          if pos_bias > 0.01
          else "Late-token focus (unusual)")
 
    # [3] Hybrid decision
    print("\n[3] HYBRID DECISION")
    print(f"  AraBERT probability : {round(bert_prob, 3)}")
    print(f"  Final hybrid score  : {round(final_prob, 3)}")
    print(f"  Prediction          : {prediction}")
    print("=" * 65)

In [ ]:

explain_prediction("الرجل كالأسد في الشجاعة")
explain_prediction("السماء صافية اليوم")
explain_prediction("قلبها كالحجارة في القسوة")
explain_prediction("ذهب الطالب إلى المدرسة")